In [ ]:
import sys
sys.path.insert(0, '/mnt/qh2-nas3/00-model/00-fb/mmseg_dino_agri')
sys.path.insert(0, '/mnt/qh2-nas3/00-model/00-fb/mmseg_unet')

import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from mmengine.config import Config
from mmengine.registry import init_default_scope
from mmengine.model import revert_sync_batchnorm
from mmseg.registry import MODELS
from mmseg.structures import SegDataSample

init_default_scope('mmseg')

# ── Model 1: DINOv3-Mask2Former (256×256, 无归一化) ──
cfg_m2f = Config.fromfile('/mnt/qh2-nas3/00-model/00-fb/mmseg_dino_agri/configs/dinov3l_m2f_agri.py')
cfg_m2f.model.pretrained = None
cfg_m2f.model.train_cfg = None
model_m2f = MODELS.build(cfg_m2f.model)
ckpt_m2f = torch.load('/mnt/qh2-nas3/00-model/00-fb/mmseg_dino_agri/work_dirs/123/20260616_193556/iter_40000.pth', map_location='cpu', weights_only=False)
model_m2f.load_state_dict(ckpt_m2f['state_dict'], strict=False)
model_m2f.dataset_meta = {'classes': ['background', 'cropland'], 'palette': [[0, 0, 0], [34, 139, 34]]}
model_m2f.cfg = cfg_m2f
model_m2f.to('cuda:0').eval()
print('M2F loaded')

# ── Model 2: UNet (512×512, CustomNormalize) ──
cfg_unet = Config.fromfile('/mnt/qh2-nas3/00-model/00-fb/mmseg_unet/configs/unet_customAgri.py')
cfg_unet.model.pretrained = None
cfg_unet.model.train_cfg = None
model_unet = MODELS.build(cfg_unet.model)
model_unet = revert_sync_batchnorm(model_unet)
ckpt_unet = torch.load('/mnt/qh2-nas3/00-model/00-fb/mmseg_unet/work_dirs/iter_24000.pth', map_location='cpu', weights_only=False)
model_unet.load_state_dict(ckpt_unet['state_dict'], strict=False)
model_unet.dataset_meta = {'classes': ['background', 'cropland'], 'palette': [[0, 0, 0], [34, 139, 34]]}
model_unet.cfg = cfg_unet
model_unet.to('cuda:0').eval()
print('UNet loaded')

In [ ]:
fn = '20_30_SVDT_JS_JiTingJieDao_2024_08m'
IMG_PATH = '/mnt/ht2-nas2/00-model/00-jiangzf/label20000/Segmentation/img_dir/' + fn + '.png'
GT_PATH  = '/mnt/ht2-nas2/00-model/00-jiangzf/label20000/Segmentation/ann_dir/' + fn + '_mask_seg.png'

img_bgr = cv2.imread(IMG_PATH, cv2.IMREAD_COLOR)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
H, W = img_rgb.shape[:2]

gt_raw = cv2.imread(GT_PATH, cv2.IMREAD_GRAYSCALE)
gt_mask = (gt_raw == 255).astype(np.uint8)

# UNet 归一化常量 (与 CustomNormalize 一致)
MEAN = np.array([72.4085, 89.7399, 69.6123], dtype=np.float32)
STD  = np.array([32.8544, 23.9954, 23.1234], dtype=np.float32)

def infer(model, img_rgb, img_size, normalize):
    """在指定尺寸推理, 输出 upsample 回原图尺寸的 mask"""
    img_r = cv2.resize(img_rgb, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    if normalize:
        img_in = (img_r.astype(np.float32) - MEAN) / STD
    else:
        img_in = img_r.astype(np.float32)
    tensor = torch.from_numpy(img_in).permute(2, 0, 1).cuda()  # (3, S, S)
    ds = SegDataSample()
    ds.set_metainfo({'img_shape': (img_size, img_size), 'ori_shape': (img_size, img_size),
                     'pad_shape': (img_size, img_size), 'scale_factor': (1.0, 1.0),
                     'flip': False, 'flip_direction': None})
    with torch.no_grad():
        r = model.test_step({'inputs': [tensor], 'data_samples': [ds]})
    pred = r[0].pred_sem_seg.data.cpu().numpy().squeeze(0)
    return cv2.resize(pred.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST)

# M2F: 256×256, 无归一化
pred_m2f  = infer(model_m2f,  img_rgb, 256, normalize=False)
# UNet: 512×512, 有归一化
pred_unet = infer(model_unet, img_rgb, 512, normalize=True)

# IoU
def calc_iou(pred, gt):
    tp = ((pred == 1) & (gt == 1)).sum()
    fp = ((pred == 1) & (gt == 0)).sum()
    fn = ((pred == 0) & (gt == 1)).sum()
    return tp / (tp + fp + fn + 1e-8)

iou_m2f  = calc_iou(pred_m2f,  gt_mask)
iou_unet = calc_iou(pred_unet, gt_mask)
print(f'M2F  IoU = {iou_m2f:.4f}')
print(f'UNet IoU = {iou_unet:.4f}')

In [ ]:
palette = np.array([[0, 0, 0], [34, 139, 34]], dtype=np.uint8)

# ── 对比误差图 ──
m2f_ok  = (pred_m2f  == gt_mask)
unet_ok = (pred_unet == gt_mask)
cmp_map = np.zeros((H, W, 3), dtype=np.uint8)
cmp_map[m2f_ok  &  unet_ok] = [0,   200, 0]    # 绿: 两个都对
cmp_map[m2f_ok  & ~unet_ok] = [0,   100, 255]  # 蓝: 只有 M2F 对
cmp_map[~m2f_ok &  unet_ok] = [255, 165, 0]    # 橙: 只有 UNet 对
cmp_map[~m2f_ok & ~unet_ok] = [255, 0,   0]    # 红: 两个都错

# ── 绘制 2×2 ──
fig, axes = plt.subplots(2, 2, figsize=(14, 14))

# Row 0: 原图 | 误差对比
axes[0, 0].imshow(img_rgb)
axes[0, 0].set_title('Original Image', fontsize=14, fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(img_rgb)
axes[0, 1].imshow(cmp_map, alpha=0.5)
axes[0, 1].set_title('Error Comparison', fontsize=14, fontweight='bold')
axes[0, 1].axis('off')
legend = [
    Patch(facecolor='#00c800', label='Both correct'),
    Patch(facecolor='#0064ff', label='Only M2F correct'),
    Patch(facecolor='#ffa500', label='Only UNet correct'),
    Patch(facecolor='#ff0000', label='Both wrong'),
]
axes[0, 1].legend(handles=legend, loc='lower left', fontsize=9, framealpha=0.85)

# Row 1: M2F | UNet
axes[1, 0].imshow(img_rgb)
axes[1, 0].imshow(palette[pred_m2f], alpha=0.5)
axes[1, 0].contour(gt_mask, levels=[0.5], colors='white', linewidths=1.5, linestyles='--')
axes[1, 0].set_title(f'DINOv3-M2F (256)  IoU={iou_m2f:.3f}', fontsize=14, fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].imshow(img_rgb)
axes[1, 1].imshow(palette[pred_unet], alpha=0.5)
axes[1, 1].contour(gt_mask, levels=[0.5], colors='white', linewidths=1.5, linestyles='--')
axes[1, 1].set_title(f'UNet (512)  IoU={iou_unet:.3f}', fontsize=14, fontweight='bold')
axes[1, 1].axis('off')

plt.suptitle(f'M2F vs UNet  —  {fn}', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()